# Exam 1

#### Anna G. Castillo

In [17]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

#### Step 1:

In [18]:
df_insurance = pd.read_csv('auto_insurance_python.csv')
df_insurance = df_insurance.dropna(subset=['State'])

##### Question 1

In [19]:
print("Average Income:", round(df_insurance['Income'].mean(), 4))

policies_by_state = df_insurance[df_insurance['State'].isin(['Arizona', 'Oregon'])].groupby('State')['Number of Policies'].mean()
print("Average Number of Policies (Arizona and Oregon):")
print(policies_by_state)

zero_income_count = (df_insurance['Income'] == 0).sum()
zero_income_pct = zero_income_count / len(df_insurance) * 100
print("Customers with Income = 0:", zero_income_count)
print("Percentage of customers with Income = 0:", round(zero_income_pct, 2))

Average Income: 39005.9103
Average Number of Policies (Arizona and Oregon):
State
Arizona    3.200514
Oregon     2.880200
Name: Number of Policies, dtype: float64
Customers with Income = 0: 473
Percentage of customers with Income = 0: 23.83


#### Step 2:

##### Step 2.1

In [20]:
def bin_policies(n):
    if n > 5:
        return 3
    elif n > 1:
        return 2
    else:
        return 1

df_insurance['Number of Policies'] = df_insurance['Number of Policies'].apply(bin_policies)

##### Step 2.2

In [21]:
df_insurance = df_insurance.drop(columns=['Customer ID'])

##### Step 2.3

In [22]:
df_insurance = pd.get_dummies(df_insurance, columns=['State', 'Coverage', 'Vehicle Class'], drop_first=True, dtype=int)

##### Step 2.4

In [23]:
scale_cols = ['Income', 'Months Since Policy Inception', 'Customer Lifetime Revenue']
stScaler = StandardScaler()
df_insurance[scale_cols] = stScaler.fit_transform(df_insurance[scale_cols])

##### Question 2

In [24]:
print("Frequency of each bin:")
print(df_insurance['Number of Policies'].value_counts())
print("Max of standardized Income:", round(df_insurance['Income'].max(), 3))
print("Number of attributes:", df_insurance.shape[1])

Frequency of each bin:
Number of Policies
2    914
1    718
3    353
Name: count, dtype: int64
Max of standardized Income: 1.995
Number of attributes: 16


#### Step 3:

In [25]:
df_prediction = df_insurance[df_insurance['Response'].isnull()].copy()
df_insurance = df_insurance.dropna(subset=['Response'])

##### Question 3

In [26]:
print("Observations in df_insurance:", len(df_insurance))
print("Observations in df_prediction:", len(df_prediction))

Observations in df_insurance: 1960
Observations in df_prediction: 25


#### Step 4:

In [27]:
X = df_insurance.drop(columns=['Response'])
y = df_insurance['Response']

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, test_size=0.3, random_state=1)

modelNN = MLPClassifier(hidden_layer_sizes=(5,), random_state=1)
modelNN.fit(X_train, y_train)
predicted_test = modelNN.predict(X_test)

##### Question 4

In [28]:
print("Accuracy:", round(accuracy_score(y_test, predicted_test), 4))
print("Precision:", round(precision_score(y_test, predicted_test, pos_label=1), 4))
print("Recall:", round(recall_score(y_test, predicted_test, pos_label=1), 4))
print("Confusion Matrix (rows = actual [1, 0], columns = predicted [1, 0]):")
print(confusion_matrix(y_test, predicted_test, labels=[1, 0]))

Accuracy: 0.5136
Precision: 0.5142
Recall: 0.67
Confusion Matrix (rows = actual [1, 0], columns = predicted [1, 0]):
[[199  98]
 [188 103]]


#### Step 5:

In [29]:
X_prediction = df_prediction.drop(columns=['Response'])
predicted_response = modelNN.predict(X_prediction)

##### Question 5

In [30]:
print("Predicted responses in df_prediction:")
print(pd.Series(predicted_response).value_counts())

Predicted responses in df_prediction:
1.0    18
0.0     7
Name: count, dtype: int64


#### Step 6:

In [31]:
prob_class1 = modelNN.predict_proba(X_prediction)[:, 1]
df_prediction['Prob_Class1'] = prob_class1

##### Question 6

In [32]:
print("Predicted Class=1 at 0.5 cutoff:", (prob_class1 >= 0.5).sum())
print("Predicted Class=1 at 0.6 cutoff:", (prob_class1 >= 0.6).sum())

Predicted Class=1 at 0.5 cutoff: 18
Predicted Class=1 at 0.6 cutoff: 4
